# GPGPU Tasks 2026 — Kaggle Runner (CUDA)

**Settings справа:** Accelerator = **GPU T4 x2** (или P100), Internet = **on**.

Workflow: правишь код локально → `git push` в ветку `task03` → здесь ячейка 4 (pull + make + запуск).

In [ ]:
# 1. GPU
!nvidia-smi

In [ ]:
# 2. [1 раз за сессию] Клонирование + зависимости + первая сборка
BRANCH = "task03"
%cd /kaggle/working
!rm -rf /kaggle/working/GPGPUTasks2026
!git clone -b $BRANCH https://github.com/BOGOMOLOV-ARSENIQ/GPGPUTasks2026.git /kaggle/working/GPGPUTasks2026
!chmod +x /kaggle/working/GPGPUTasks2026/scripts/linux/*.sh
!/kaggle/working/GPGPUTasks2026/scripts/linux/install_vulkan_sdk.sh --no-validation-layers
!sudo apt-get update -qq && sudo apt-get install -yqq libgtest-dev libgraphicsmagick++1-dev

In [ ]:
# 3. [1 раз за сессию] Конфигурация CMake (75 = T4)
# На Kaggle nvcc не лежит в PATH, и старый FindCUDA его не находит - ищем тулкит руками
import glob, os
cands = [d for d in ["/usr/local/cuda"] + sorted(glob.glob("/usr/local/cuda-*"), reverse=True) if os.path.exists(d + "/bin/nvcc")]
assert cands, "CUDA toolkit (nvcc) не найден в /usr/local/cuda*"
CUDA_ROOT = cands[0]
os.environ["PATH"] = CUDA_ROOT + "/bin:" + os.environ["PATH"]
print("CUDA toolkit:", CUDA_ROOT)
!nvcc --version | tail -2
!rm -rf /kaggle/working/GPGPUTasks2026/build && mkdir -p /kaggle/working/GPGPUTasks2026/build
%cd /kaggle/working/GPGPUTasks2026/build
!cmake -DCMAKE_BUILD_TYPE=Release -DGPU_CUDA_SUPPORT=ON -DCUDA_TOOLKIT_ROOT_DIR={CUDA_ROOT} -DCMAKE_CUDA_COMPILER={CUDA_ROOT}/bin/nvcc -DCMAKE_CUDA_ARCHITECTURES=75 ..

In [ ]:
# 4. [Каждый раз после git push] Подтянуть код, пересобрать, запустить
%cd /kaggle/working/GPGPUTasks2026
!git fetch -q origin && git reset -q --hard origin/$BRANCH && git log --oneline -1
%cd /kaggle/working/GPGPUTasks2026/build
!make -j$(nproc) 2>&1 | tail -20

print("\n\033[94m=== main_matrix_transpose ===\033[0m")
!./main_matrix_transpose 0
# умножение пока не реализовано, а его CPU-эталон идёт ~80 сек - включить, когда дойдём до части 2
# print("\n\033[94m=== main_matrix_multiply ===\033[0m")
# !./main_matrix_multiply 0